# Robust Music Watermarking: audiowmark на FMA-small

**Автор частини:** Богдана Зубрицька. Командний проєкт курсу «Штучний інтелект» (УКУ).

**Що робить цей ноутбук.** Він перевіряє, чи витримує водяний знак у музиці типові перетворення сигналу. Метод: **audiowmark**, класичний spread-spectrum з відкритим кодом. Тестова вибірка: 80 треків FMA-small (по 10 на кожен із 8 жанрів). У мітку вбудовується 16 біт, це спільна домовленість команди, щоб результати можна було порівнювати з AudioSeal і WavMark.

**Мої атаки (час і динаміка):** зміна швидкості, time-stretch, pitch shift, зміна гучності, обрізання.

**Що виходить на виході:**
- `results_audiowmark.csv`: основний прогін, 80 треків × 16 варіантів = 1280 рядків;
- `results_audiowmark_detect_speed.csv`: додатковий експеримент із режимом `--detect-speed`, 160 рядків;
- формат обох файлів: `method, attack, params, genre, track_id, detected, snr, bit_acc`.

**Як запускати.** Ноутбук розрахований на Google Colab, файли зберігаються на Google Drive (папка `wm`). Довгі прогони (розділи 8 і 10) вміють продовжуватися: якщо результат уже є на Drive, вони пропустять оброблені треки.

## 0. Підготовка середовища

Підключаємо Google Drive (там лежать результати, бо файли в `/content` зникають після завершення сесії Colab) і ставимо бібліотеки для роботи з аудіо.

In [1]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q librosa soundfile pandas

Mounted at /content/drive


## 1. Дані: FMA-small

FMA-small це 8000 треків по 30 секунд, 8 жанрів, по 1000 на жанр. Завантажуємо два архіви:
- `fma_metadata.zip` (342 МБ): таблиця `tracks.csv` з жанрами та ідентифікаторами;
- `fma_small.zip` (7,2 ГБ): самі аудіофайли (mp3).

Джерело: https://github.com/mdeff/fma. Метадані під ліцензією CC BY 4.0, аудіо під ліцензіями, які обрали виконавці (переважно Creative Commons). Аудіо в репозиторій не додаємо.

Після завантаження звіряємо контрольну суму архіву з тією, що вказана в документації датасету.

In [2]:
!wget -q https://os.unil.cloud.switch.ch/fma/fma_metadata.zip -O fma_metadata.zip
!unzip -q fma_metadata.zip

In [3]:
!wget -q https://os.unil.cloud.switch.ch/fma/fma_small.zip -O fma_small.zip

In [4]:
!echo "ade154f733639d52e35e32f5593efe5be76c6d70 fma_small.zip" | sha1sum -c -

fma_small.zip: OK


`OK` означає, що архів завантажився цілим.

## 2. Тестова вибірка: 80 треків

Беремо треки з підмножини `small`, сортуємо за `track_id` і для кожного жанру беремо **перші 10**. Правило детерміноване, тому в усіх членів команди вибірка збігається. Список зберігаємо у `selected_80.csv` і копіюємо на Drive.

In [5]:
import pandas as pd
t = pd.read_csv("fma_metadata/tracks.csv", index_col=0, header=[0, 1])
small = t[t[("set", "subset")] == "small"]
small = small[[("track", "genre_top")]].droplevel(0, axis=1).reset_index()
small.columns = ["track_id", "genre"]
small = small.sort_values("track_id")
sel = small.groupby("genre").head(10)
sel.to_csv("selected_80.csv", index=False)
print(len(sel), sel["genre"].nunique())
print(sel.head())

80 8
   track_id    genre
0         2  Hip-Hop
1         5  Hip-Hop
2        10      Pop
3       140     Folk
4       141     Folk


In [6]:
import os, shutil
os.makedirs("/content/drive/MyDrive/wm/wav", exist_ok=True)
shutil.copy("selected_80.csv", "/content/drive/MyDrive/wm/selected_80.csv")

'/content/drive/MyDrive/wm/selected_80.csv'

Вивід `80 8` означає 80 треків і 8 жанрів, як і очікувалось.

## 3. Підготовка аудіо

Витягуємо з великого архіву лише 80 потрібних mp3 і конвертуємо їх у WAV: **mono, 44,1 кГц**. Один формат для всіх методів прибирає зайву змінну при порівнянні. Зауваження: початкові файли FMA вже були в mp3, тож «оригінал» тут це декодований mp3.

In [7]:
import pandas as pd, subprocess
ids = pd.read_csv("selected_80.csv")["track_id"]
paths = [f"fma_small/{str(i).zfill(6)[:3]}/{str(i).zfill(6)}.mp3" for i in ids]
subprocess.run(["unzip", "-q", "-o", "fma_small.zip"] + paths, check=True)
print(len(paths), "запитано")

80 запитано


In [8]:
!find fma_small -name "*.mp3" | wc -l

80


In [9]:
import os
out = "/content/drive/MyDrive/wm/wav"
os.makedirs(out, exist_ok=True)
for i in ids:
    s = str(i).zfill(6)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", f"fma_small/{s[:3]}/{s}.mp3",
                    "-ac", "1", "-ar", "44100", f"{out}/{s}.wav"], check=True)
print("готово")

готово


Перевірка: запитано 80 файлів, знайдено 80, конвертація завершилась без помилок.

## 4. Встановлення audiowmark

audiowmark потрібно зібрати з вихідного коду (https://github.com/swesterfeld/audiowmark). Збірка займає кілька хвилин і потрібна після кожного нового запуску сесії Colab. Вивід збірки очищено, бо він дуже довгий.

In [10]:
!apt-get -qq install -y autoconf automake libtool pkg-config libfftw3-dev libsndfile1-dev libgcrypt20-dev libzita-resampler-dev libmpg123-dev
!git clone -q https://github.com/swesterfeld/audiowmark.git
%cd audiowmark
!./autogen.sh && make -j4 && make install
%cd /content
!audiowmark --help | head -40

Selecting previously unselected package libfftw3-long3:amd64.
(Reading database ... 122807 files and directories currently installed.)
Preparing to unpack .../00-libfftw3-long3_3.3.10-1ubuntu3_amd64.deb ...
Unpacking libfftw3-long3:amd64 (3.3.10-1ubuntu3) ...
Selecting previously unselected package libfftw3-quad3:amd64.
Preparing to unpack .../01-libfftw3-quad3_3.3.10-1ubuntu3_amd64.deb ...
Unpacking libfftw3-quad3:amd64 (3.3.10-1ubuntu3) ...
Selecting previously unselected package libfftw3-single3:amd64.
Preparing to unpack .../02-libfftw3-single3_3.3.10-1ubuntu3_amd64.deb ...
Unpacking libfftw3-single3:amd64 (3.3.10-1ubuntu3) ...
Selecting previously unselected package libfftw3-bin.
Preparing to unpack .../03-libfftw3-bin_3.3.10-1ubuntu3_amd64.deb ...
Unpacking libfftw3-bin (3.3.10-1ubuntu3) ...
Selecting previously unselected package libfftw3-dev:amd64.
Preparing to unpack .../04-libfftw3-dev_3.3.10-1ubuntu3_amd64.deb ...
Unpacking libfftw3-dev:amd64 (3.3.10-1ubuntu3) ...
Selecting 

## 5. Перевірка на одному треку

Перш ніж запускати все на 80 треках, перевіряємо базову роботу методу на першому треку:
1. вбудовуємо 16-бітну мітку `a5c3` (режим `--short 16`, сила за замовчуванням 10);
2. читаємо мітку з файлу з міткою;
3. контроль: читаємо мітку з оригіналу, де її немає.

In [11]:
import glob
wav = sorted(glob.glob("/content/drive/MyDrive/wm/wav/*.wav"))[0]
print(wav)
!audiowmark add --short 16 {wav} /content/test_wm.wav a5c3

/content/drive/MyDrive/wm/wav/000002.wav
Input:        /content/drive/MyDrive/wm/wav/000002.wav
Output:       /content/test_wm.wav
Message:      a5c3
Strength:     10

Time:         0:29
Sample Rate:  44100
Channels:     1
Data Blocks:  0


In [12]:
!audiowmark get --short 16 /content/test_wm.wav

pattern  0:00 a5c3 1.399 0.276 CLIP-B


In [13]:
!audiowmark get --short 16 {wav}

**Результат.** З файлу з міткою прочитано `a5c3`, повідомлення збігається. У рядку є ще числа та позначка `CLIP-B`: це службова інформація audiowmark, яку ми в аналізі не використовуємо.

Контроль на оригіналі нічого не вивів, тобто мітки там не знайдено. Це очікувана поведінка (при відсутності мітки детектор мовчить), але це перевірка лише на одному треку.

## 6. Функції для експерименту

Тут зібрані всі функції та налаштування:
- `wm_embed` / `wm_detect`: вбудувати мітку та прочитати її (повертає hex-код або `None`);
- `apply_attack`: атаки на сигнал;
- `snr_db`: якість звуку після вбудовування, `SNR = 10·log10( Σx² / Σ(x − x_w)² )`, де `x` оригінал, `x_w` трек із міткою (чим більше, тим слабше змінився сигнал);
- `bit_acc`: частка збігів бітів між вбудованим і прочитаним повідомленням.

**Правило «виявлено»:** `bit accuracy ≥ 14/16`. Для випадкового 16-бітного рядка ймовірність такого збігу близько 0,2%. Якщо мітку не знайдено зовсім, `detected = False`, а `bit_acc` лишається порожнім.

**Атаки (однакові параметри мають бути у всіх методів):**
- `speed`: зміна швидкості разом із висотою (передискретизація), 0,9×, 0,95×, 1,05×, 1,1×;
- `time_stretch`: зміна темпу без зміни висоти, ті самі коефіцієнти;
- `pitch`: зсув висоти на −2, −1, +1, +2 півтони;
- `gain`: зміна гучності на −6 та +6 дБ (піки обрізаються на межі −1…1);
- `crop`: залишаємо першу половину треку;
- `none`: без атаки (базова лінія).

In [14]:
import csv, os, re, subprocess, tempfile, time
import numpy as np, pandas as pd, soundfile as sf, librosa

BITS, MSG = 16, "a5c3"       # довжина повідомлення і його значення (hex)
THRESHOLD = 14 / 16          # detected = bit accuracy >= 14/16
WAV_DIR = "/content/drive/MyDrive/wm/wav"

ATTACKS = {"none": [None],
           "speed": [0.9, 0.95, 1.05, 1.1],
           "time_stretch": [0.9, 0.95, 1.05, 1.1],
           "pitch": [-2, -1, 1, 2],
           "gain": [-6, 6],
           "crop": [0.5]}

def wm_embed(src, dst):
    subprocess.run(["audiowmark", "add", "--short", str(BITS), src, dst, MSG],
                   check=True, capture_output=True)

def wm_detect(path, extra=()):
    """Повертає прочитаний hex-код або None, якщо мітку не знайдено."""
    out = subprocess.run(["audiowmark", "get", "--short", str(BITS), *extra, path],
                         capture_output=True, text=True).stdout
    m = re.search(r"^pattern\s+\S+\s+([0-9a-f]+)", out, re.M)
    return m.group(1) if m else None

def apply_attack(name, x, sr, p):
    if name == "none": return x
    if name == "speed": return librosa.resample(x, orig_sr=sr, target_sr=int(sr / p))
    if name == "time_stretch": return librosa.effects.time_stretch(x, rate=p)
    if name == "pitch": return librosa.effects.pitch_shift(x, sr=sr, n_steps=p)
    if name == "gain": return np.clip(x * 10 ** (p / 20), -1, 1)
    if name == "crop": return x[: int(len(x) * p)]
    raise ValueError(f"unknown attack: {name}")

def snr_db(x, y):
    n = min(len(x), len(y)); x, y = x[:n], y[:n]
    return 10 * np.log10(np.sum(x**2) / np.sum((x - y) ** 2))

def bit_acc(a_hex, b_hex):
    a = bin(int(a_hex, 16))[2:].zfill(BITS); b = bin(int(b_hex, 16))[2:].zfill(BITS)
    return sum(i == j for i, j in zip(a, b)) / BITS

## 7. Пробний прогін атак на одному треку

Швидка перевірка, що атаки працюють, на треку з розділу 5 (по два значення для кожної атаки). Потім контроль коректності самого коду: **тотожні атаки** (швидкість 1,0×, темп 1,0×, зсув 0 півтонів) нічого не змінюють, тож мітка мусить читатися.

In [15]:
xw, sr = sf.read("/content/test_wm.wav")
pilot = {"none": [None], "speed": [0.9, 1.1], "time_stretch": [0.9, 1.1],
         "pitch": [-2, 2], "gain": [-6, 6], "crop": [0.5]}
for name, params in pilot.items():
    for p in params:
        sf.write("/content/att.wav", apply_attack(name, xw, sr, p), sr)
        print(name, p, "->", wm_detect("/content/att.wav"))

none None -> a5c3
speed 0.9 -> None
speed 1.1 -> None
time_stretch 0.9 -> None
time_stretch 1.1 -> None
pitch -2 -> None
pitch 2 -> None
gain -6 -> a5c3
gain 6 -> a5c3
crop 0.5 -> a5c3


In [16]:
for name, p in [("speed", 1.0), ("time_stretch", 1.0), ("pitch", 0)]:
    sf.write("/content/att.wav", apply_attack(name, xw, sr, p), sr)
    print(name, p, "->", wm_detect("/content/att.wav"))

speed 1.0 -> a5c3
time_stretch 1.0 -> a5c3
pitch 0 -> a5c3


**Результат.** На цьому треку мітка вижила без атаки, при зміні гучності (±6 дБ) та при обрізанні. Після зміни швидкості, темпу та висоти її знайти не вдалося (`None`).

Контроль із тотожними атаками повертає `a5c3`, тож функції атак працюють коректно, і `None` в попередньому виводі означає саме те, що мітка знищена атакою, а не помилку коду.

## 8. Основний прогін: 80 треків × 16 варіантів

Для кожного треку: вбудовуємо мітку, рахуємо SNR, застосовуємо кожну атаку з кожним параметром і пробуємо прочитати мітку. Кожен результат записується в CSV одразу, а вже оброблені комбінації `(трек, атака, параметр)` пропускаються, тож перерваний прогін можна безпечно запустити ще раз. На один трек іде приблизно 25–30 секунд.

У параметрі `none` значення параметра порожнє. Для перевірки «вже оброблено» воно перетворюється на слово `None`, інакше ці рядки дублювалися б.

In [17]:
OUT = "/content/drive/MyDrive/wm/results_audiowmark.csv"
sel = pd.read_csv("selected_80.csv")

done = set()
if os.path.exists(OUT):
    prev = pd.read_csv(OUT, dtype=str, keep_default_na=False)
    prev["params"] = prev["params"].replace("", "None")
    done = set(zip(prev.track_id, prev.attack, prev.params))
new = not os.path.exists(OUT)

with open(OUT, "a", newline="") as f:
    w = csv.writer(f)
    if new:
        w.writerow(["method", "attack", "params", "genre", "track_id", "detected", "snr", "bit_acc"])
    for _, r in sel.iterrows():
        tid = int(r.track_id)
        todo = [(n, p) for n, ps in ATTACKS.items() for p in ps
                if (str(tid), n, str(p)) not in done]
        if not todo:
            continue
        t0 = time.time()
        try:
            src = f"{WAV_DIR}/{tid:06d}.wav"
            with tempfile.TemporaryDirectory() as tmp:
                wm = f"{tmp}/wm.wav"
                wm_embed(src, wm)
                x, sr = sf.read(src)
                xw, _ = sf.read(wm)
                s = round(snr_db(x, xw), 2)
                for n, p in todo:
                    att = f"{tmp}/att.wav"
                    sf.write(att, apply_attack(n, xw, sr, p), sr)
                    got = wm_detect(att)
                    acc = round(bit_acc(MSG, got), 3) if got else ""
                    det = bool(got) and acc >= THRESHOLD
                    w.writerow(["audiowmark", n, p, r.genre, tid, det, s, acc])
            f.flush()
            print(tid, r.genre, f"{time.time() - t0:.0f} с")
        except Exception as e:
            print("ПОМИЛКА", tid, e)

Журнал прогону (час на кожен трек) тут не показано, бо прогін виконувався кількома запусками. Правильність результату перевіряємо за самим файлом у наступному розділі.

## 9. Перевірка файлу і основні результати

Перевіряємо цілісність: кількість рядків, кількість треків, рядків на трек, дублікати, баланс жанрів. Далі дивимося частку виявлень для кожної атаки.

In [18]:
d = pd.read_csv(OUT, keep_default_na=False)
print(len(d), d.track_id.nunique())
print(d.groupby("track_id").size().value_counts())
print("дублікатів:", d.duplicated(["track_id", "attack", "params"]).sum())
print(d.drop_duplicates("track_id").genre.value_counts())
print(d.groupby("attack").detected.mean().round(2))

1280 80
16    80
Name: count, dtype: int64
дублікатів: 0
genre
Hip-Hop          10
Pop              10
Folk             10
Experimental     10
Rock             10
International    10
Electronic       10
Instrumental     10
Name: count, dtype: int64
attack
crop            1.0
gain            1.0
none            1.0
pitch           0.0
speed           0.0
time_stretch    0.0
Name: detected, dtype: float64


**Цілісність.** Файл містить 1280 рядків: 80 треків × 16 варіантів. На кожен трек рівно 16 рядків, дублікатів немає, у кожному жанрі по 10 треків.

**Робастність (частка треків, де мітку виявлено):**
- без атаки, зміна гучності (±6 дБ) та обрізання до 50%: **1,0**, мітку знайдено в усіх 80 треках;
- зміна швидкості, time-stretch і pitch shift: **0,0**, мітку не знайдено в жодному треку, навіть при м'яких параметрах (5% швидкості, 1 півтон).

**Можливе пояснення (гіпотеза, не перевірена окремо).** Швидкість, темп і висота змінюють часову та частотну структуру сигналу, і детектор, який шукає мітку на очікуваній сітці, втрачає синхронізацію. Гучність і обрізання цієї структури не чіпають. Узгоджується з цим те, що в розділі 10 режим корекції швидкості повертає мітку при зміні швидкості.

**Жанри.** Для цих атак відмінностей між жанрами немає: усі значення дорівнюють 0 або 1. Тому порівнювати жанри за робастністю тут нічим. Це може змінитися для інших атак (стиснення, шум), які робить решта команди.

In [19]:
s = d[d.attack == "none"].copy()
s["snr"] = s["snr"].astype(float)
print(s.groupby("genre").snr.agg(["mean", "min", "max"]).round(1))
print("всього:", round(s.snr.mean(), 1), round(s.snr.min(), 1), round(s.snr.max(), 1))

               mean   min   max
genre                          
Electronic     33.8  26.6  46.9
Experimental   30.4  26.2  51.0
Folk           30.7  29.0  34.8
Hip-Hop        29.9  25.5  34.6
Instrumental   31.2  26.5  35.5
International  30.9  28.0  33.8
Pop            30.2  26.8  37.0
Rock           30.7  26.8  36.2
всього: 31.0 25.5 51.0


**Якість звуку (SNR після вбудовування, без атаки).** Середнє по всіх 80 треках 31,0 дБ, діапазон від 25,5 до 51,0 дБ. За жанрами середнє лежить у вузьких межах від 29,9 дБ (Hip-Hop) до 33,8 дБ (Electronic).

Застереження:
- розкид усередині одного жанру більший за різницю між жанрами (наприклад, в Electronic від 26,6 до 46,9 дБ), тож жанровий ефект слабкий;
- окремі великі значення (46,9 і 51,0) підтягують середнє, тому для жанрів корисно додатково дивитися медіану;
- на жанр припадає лише 10 треків, тому висновки попередні;
- SNR показує, наскільки змінився сигнал, а не наскільки це чутно. Слухових тестів ми не проводили, тому стверджувати, що мітка непомітна, не можна.

## 10. Додатковий експеримент: режим `--detect-speed`

audiowmark має опцію `--detect-speed`, яка виявляє й виправляє різницю в швидкості відтворення. В основному прогоні вона **вимкнена**: AudioSeal і WavMark такого режиму не мають, і порівняння має бути на однакових умовах. Тут ми окремо перевіряємо, скільки дає ця опція.

**Крок 1: тест на одному треку.** Опція повертає мітку при зміні швидкості (режим навіть показує знайдену швидкість, 0,9 і 1,1). Для `time_stretch` і `pitch` вона мітку не повернула.

In [20]:
xw, sr = sf.read("/content/test_wm.wav")
for name, p in [("speed", 0.9), ("speed", 1.1), ("time_stretch", 0.9), ("pitch", 2)]:
    sf.write("/content/att.wav", apply_attack(name, xw, sr, p), sr)
    out = subprocess.run(["audiowmark", "get", "--short", "16", "--detect-speed", "/content/att.wav"],
                         capture_output=True, text=True)
    print(name, p, "->", repr(out.stdout))

speed 0.9 -> 'speed 0.900032\npattern  0:00 a5c3 1.400 0.273 CLIP-B-SPEED\n'
speed 1.1 -> 'speed 1.100063\npattern  0:00 a5c3 1.400 0.273 CLIP-B-SPEED\n'
time_stretch 0.9 -> ''
pitch 2 -> ''


**Крок 2: прогін на всіх 80 треках.** Обмежуємось значеннями швидкості 0,9× та 1,1×, бо опція повільніша: у вимірі на одному виклику 12,9 с проти 4,1 с без неї. Для `time_stretch` і `pitch` на всій вибірці цей режим не перевіряли, бо тест на одному треку нічого не дав. Результати пишемо в окремий файл, а метод у ньому підписано `audiowmark+detect-speed`.

In [21]:
OUT2 = "/content/drive/MyDrive/wm/results_audiowmark_detect_speed.csv"
PARAMS = [0.9, 1.1]

done = set()
exists = os.path.exists(OUT2)
if exists:
    prev2 = pd.read_csv(OUT2, dtype=str, keep_default_na=False)
    done = set(zip(prev2.track_id, prev2.attack, prev2.params))

with open(OUT2, "a", newline="") as f:
    w = csv.writer(f)
    if not exists:
        w.writerow(["method", "attack", "params", "genre", "track_id", "detected", "snr", "bit_acc"])
    for _, r in sel.iterrows():
        tid = int(r.track_id)
        todo = [p for p in PARAMS if (str(tid), "speed", str(p)) not in done]
        if not todo:
            continue
        t0 = time.time()
        try:
            src = f"{WAV_DIR}/{tid:06d}.wav"
            with tempfile.TemporaryDirectory() as tmp:
                wm = f"{tmp}/wm.wav"
                wm_embed(src, wm)
                x, sr = sf.read(src)
                xw, _ = sf.read(wm)
                s = round(snr_db(x, xw), 2)
                for p in todo:
                    att = f"{tmp}/att.wav"
                    sf.write(att, apply_attack("speed", xw, sr, p), sr)
                    got = wm_detect(att, extra=("--detect-speed",))
                    acc = round(bit_acc(MSG, got), 3) if got else ""
                    det = bool(got) and acc >= THRESHOLD
                    w.writerow(["audiowmark+detect-speed", "speed", p, r.genre, tid, det, s, acc])
            f.flush()
            print(tid, r.genre, f"{time.time() - t0:.0f} с")
        except Exception as e:
            print("ПОМИЛКА", tid, e)

**Крок 3: перевірка файлу і порівняння з основним прогоном.**

In [22]:
d2 = pd.read_csv(OUT2, keep_default_na=False)
print(len(d2), d2.track_id.nunique())
print(d2.groupby("track_id").size().value_counts())
print("дублікатів:", d2.duplicated(["track_id", "attack", "params"]).sum())
print(d2.groupby("params").detected.mean().round(2))

160 80
2    80
Name: count, dtype: int64
дублікатів: 0
params
0.9    1.0
1.1    1.0
Name: detected, dtype: float64


In [23]:
a = pd.read_csv(OUT, keep_default_na=False)
a = a[(a.attack == "speed") & (a.params.isin(["0.9", "1.1"]))]
b = pd.read_csv(OUT2, keep_default_na=False)

cmp = pd.concat([a, b])
print(cmp.groupby(["method", "params"]).detected.mean().round(2))

bb = b[b.bit_acc != ""].copy()
bb["bit_acc"] = bb.bit_acc.astype(float)
print("середній bit_acc з --detect-speed:", round(bb.bit_acc.mean(), 3), "| мін:", bb.bit_acc.min())

method                   params
audiowmark               0.9       0.0
                         1.1       0.0
audiowmark+detect-speed  0.9       1.0
                         1.1       1.0
Name: detected, dtype: float64
середній bit_acc з --detect-speed: 1.0 | мін: 1.0


**Результат.** Файл цілий: 160 рядків (80 треків × 2 значення швидкості), дублікатів немає.

- Без `--detect-speed` частка виявлень при швидкості 0,9× і 1,1× дорівнює **0,0**.
- З `--detect-speed` вона дорівнює **1,0** в обох випадках, а bit accuracy у всіх рядках **1,0** (і середнє, і мінімум): мітка читається без жодної помилки в бітах.

Тобто зміну швидкості на 10% на цій вибірці ця опція повністю компенсує.

Обмеження: перевірено лише 0,9× і 1,1× (значень 0,95× і 1,05× у цьому експерименті немає), а для `time_stretch` і `pitch` опція перевірена тільки на одному треку, де вона не допомогла.

## 11. Підсумки та обмеження

**Головні висновки (audiowmark, 16 біт, 80 треків):**
1. Мітка стабільно переживає зміну гучності (±6 дБ) і обрізання до половини треку: виявлення 100%.
2. Мітка не переживає зміну швидкості, темпу та висоти навіть у м'якій формі (5% і 1 півтон): виявлення 0%.
3. Для зміни швидкості проблему повністю знімає режим `--detect-speed`: виявлення 100% при 0,9× і 1,1×. Для темпу та висоти така допомога не спрацювала на тестовому треку.
4. Якість звуку за SNR приблизно 31 дБ у середньому, між жанрами різниця мала.

**Обмеження:**
- пілотна вибірка: 80 треків, по 10 на жанр, тому висновки за жанрами попередні;
- перевірено один метод, одне повідомлення (`a5c3`) і одну силу вбудовування (за замовчуванням 10);
- обрізання залишає лише початок треку, інші фрагменти не перевірялись;
- SNR не заміняє перцептивну оцінку, слухових тестів не було;
- контроль на відсутність хибних виявлень зроблено лише на одному треку;
- вхідні файли вже були mp3, перетворені у WAV.

**Далі:** прогнати ці самі атаки на AudioSeal і WavMark, злити CSV з результатами решти команди і побудувати таблицю «метод – атака – жанр».